### Memory

In some applications like chatbots, it is important to remember previous interactions to keep the whole context of a conversation. Memory does provide you an easy way to handle this

In [6]:
from dotenv import load_dotenv
import os

# Print the current working directory to verify where Python is looking for the .env file
print("Current working directory:", os.getcwd())

# Load the .env file
load_dotenv()

# Model name
model_name = "gpt-oss-120b"

# Print the API key (first few characters for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("API key found:", api_key[:4] + "..." + api_key[-4:])
else:
    print("No API key found")

Current working directory: d:\gitlab\da-225o-deep-learning\langchain\3_Memory
API key found: sk-R...-kEw


In [7]:
import warnings
warnings.filterwarnings('ignore')

warnings.filterwarnings('ignore', category=DeprecationWarning)


In [8]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()

history.add_user_message("hi!")
history.add_ai_message("hello my friend!")
history.messages

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hello my friend!', additional_kwargs={}, response_metadata={})]

In [9]:
from langchain.memory import ConversationBufferMemory

# ConversationBufferMemory is a memory that stores the conversation in a buffer.
# It is a simple memory that stores the conversation in a dictionary.

memory = ConversationBufferMemory()
memory.chat_memory.add_user_message("hi!")
memory.chat_memory.add_ai_message("hello my friend!")
memory.load_memory_variables({})

{'history': 'Human: hi!\nAI: hello my friend!'}

In [10]:
from langchain_openai import ChatOpenAI
from langchain.chains.conversation.base import ConversationChain


llm = ChatOpenAI(
    model=model_name,
    api_key=api_key,
    base_url="https://model-broker.aviator-model.bp.anthos.otxlab.net/v1",
    temperature=0
)
conversation = ConversationChain(
    llm=llm, verbose=True, memory=ConversationBufferMemory()
)
result = conversation.invoke(input="What is the capital of France?")
print(result["response"])



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: What is the capital of France?
AI:

> Finished chain.
The capital of France is **Paris**.  

A few fun details about the city:

- **Population & Size** – The city proper is home to about 2.1 million residents (roughly 11 million in the larger metropolitan area). It covers just 105 km² (≈ 40 mi²), which makes it one of the most densely populated capitals in the world.  
- **Geography** – Paris sits on the banks of the River Seine, which winds through the city in a gentle “U” shape. The historic heart, the Île de la Cité, is an island in the middle of the river and houses the famous Notre‑Dame cathedral.  
- **Landmarks** – The Eiffel Tower (built for the 1889 Wo

In [11]:
conversation.invoke(input="Whats the best food there?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: What is the capital of France?
AI: The capital of France is **Paris**.  

A few fun details about the city:

- **Population & Size** – The city proper is home to about 2.1 million residents (roughly 11 million in the larger metropolitan area). It covers just 105 km² (≈ 40 mi²), which makes it one of the most densely populated capitals in the world.  
- **Geography** – Paris sits on the banks of the River Seine, which winds through the city in a gentle “U” shape. The historic heart, the Île de la Cité, is an island in the middle of the river and houses the famous Notre‑Dame cathedral.  
- **Landmarks** – The Eiffel Tower (built for the 1889 World’s Fair) rises 32

{'input': 'Whats the best food there?',
 'history': 'Human: What is the capital of France?\nAI: The capital of France is **Paris**.  \n\nA few fun details about the city:\n\n- **Population & Size** – The city proper is home to about\u202f2.1\u202fmillion residents (roughly 11\u202fmillion in the larger metropolitan area). It covers just 105\u202fkm² (≈\u202f40\u202fmi²), which makes it one of the most densely populated capitals in the world.  \n- **Geography** – Paris sits on the banks of the River\u202fSeine, which winds through the city in a gentle “U” shape. The historic heart, the Île de la Cité, is an island in the middle of the river and houses the famous Notre‑Dame cathedral.  \n- **Landmarks** – The Eiffel\u202fTower (built for the 1889\u202fWorld’s\u202fFair) rises 324\u202fm (1,063\u202fft) and is visited by about\u202f7\u202fmillion tourists each year. Other must‑see sites include the Louvre Museum (home to the Mona\u202fLisa), the Arc de Triomphe, and the Sacré‑Cœur basilic

### ConversationSummaryMemory

When inputs get long, we might not want to send the whole conversation, but rather a summary.

In [13]:
from langchain.memory import ConversationBufferMemory

review = "I ordered Pizza Salami for 9.99$ and it was awesome! \
The pizza was delivered on time and was still hot when I received it. \
The crust was thin and crispy, and the toppings were fresh and flavorful. \
The Salami was well-cooked and complemented the cheese perfectly. \
The price was reasonable and I believe I got my money's worth. \
Overall, I am very satisfied with my order and I would recommend this pizza place to others."

summary_memory = ConversationBufferMemory()
summary_memory.save_context(
    {"input": "Hello, how can I help you today?"},
    {"output": "Could you analyze a review for me?"},
)
summary_memory.save_context(
    {"input": "Sure, I'd be happy to. Could you provide the review?"},
    {"output": f"{review}"},
)

In [14]:
from langchain_openai import ChatOpenAI
from langchain.chains.conversation.base import ConversationChain

conversation = ConversationChain(
    llm=llm, verbose=True, memory=summary_memory
)
result = conversation.invoke(input="Thank you very much!")
print(result["response"])




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hello, how can I help you today?
AI: Could you analyze a review for me?
Human: Sure, I'd be happy to. Could you provide the review?
AI: I ordered Pizza Salami for 9.99$ and it was awesome! The pizza was delivered on time and was still hot when I received it. The crust was thin and crispy, and the toppings were fresh and flavorful. The Salami was well-cooked and complemented the cheese perfectly. The price was reasonable and I believe I got my money's worth. Overall, I am very satisfied with my order and I would recommend this pizza place to others.
Human: Thank you very much!
AI:

> Finished chain.
You’re very welcome! I’m happy to dive into the review and break

In [15]:
memory.load_memory_variables({})

{'history': 'Human: hi!\nAI: hello my friend!'}

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory



prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | llm

long_history = InMemoryChatMessageHistory()

def get_long_history(session_id):
    return long_history

long_conversation = RunnableWithMessageHistory(
    chain,
    get_long_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "review_session"}}
inputs = [
    "Hello! I'd like to get your opinion on a restaurant review I received.",
    "Sure, I'd be happy to help! Could you share the review with me?",
    """Here's the review: I ordered Pizza Salami for 9.99$ and it was awesome! 
The pizza was delivered on time and was still hot when I received it. 
The crust was thin and crispy, and the toppings were fresh and flavorful. 
The Salami was well-cooked and complemented the cheese perfectly. 
The price was reasonable and I believe I got my money's worth. 
Overall, I am very satisfied with my order and I would recommend this pizza place to others.""",
    "What aspects of the review stood out to you?",
    "Do you think the price of 9.99$ is reasonable for this pizza?",
    "Would you recommend this restaurant based on the review?",
    "What other information would you have liked to see in the review?",
]

for i, user_input in enumerate(inputs, 1):
    result = long_conversation.invoke({"input": user_input}, config=config)
    print(f"\nExchange {i}:")
    print(f"Input: {user_input}")
    print(f"Response: {result.content}")


Exchange 1:
Input: Hello! I'd like to get your opinion on a restaurant review I received.
Response: Sure thing! I’d be happy to take a look. Could you share the review you received (or a snippet of it) and let me know what kind of feedback you’re after? For example:

* Do you want thoughts on the overall tone (e.g., friendly, professional, overly critical)?
* Are you looking for suggestions on how to improve the wording or structure?
* Is there a specific part of the review that you’re unsure about (e.g., a particular comment about the food, service, or ambiance)?

Feel free to paste the text here, and I’ll give you a detailed opinion and any recommendations you might find useful.

Exchange 2:
Input: Sure, I'd be happy to help! Could you share the review with me?
Response: Got it—let’s clear up the mix‑up! I’m here to give you feedback on **the review you received**, so I’ll need to see the actual text (or at least the parts you’re unsure about). If you’d rather have a sample review t